# ASV Diversity Analysis without scikit-bio (JupyterLite-ready)

This notebook reproduces your workflow **without `scikit-bio`**, using **NumPy/Pandas/Matplotlib** and (optionally) **scikit-learn**. 

It computes **alpha diversity** (richness, Shannon), **Bray–Curtis** distances, performs **PCoA**, and **saves plots to files** (no interactive windows).

## How to use in JupyterLite
1. Upload your `asv_table.tsv` (rows = ASVs, columns = samples)
2. Run the cells top-to-bottom.
3. Download the saved figures from the left file browser: `observed_richness.png`, `shannon_diversity_boxplot.png`, `pcoa_braycurtis.png`.

In [1]:
# ---------------------------------------------
# Imports & environment
# ---------------------------------------------
import sys, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Try scikit-learn if available (JupyterLite may or may not have it)
try:
    from sklearn.metrics import pairwise_distances
    from sklearn.manifold import MDS
    HAVE_SKLEARN = True
except Exception:
    HAVE_SKLEARN = False

print(f'Python: {sys.version.split()[0]}')
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('Matplotlib:', plt.matplotlib.__version__)
print('scikit-learn available:', HAVE_SKLEARN)

# Ensure plots are not displayed inline accidentally
plt.ioff()


Python: 3.13.2
NumPy: 2.2.5
Pandas: 2.3.2
Matplotlib: 3.8.4
scikit-learn available: True


In [2]:
# ---------------------------------------------
# Load ASV/OTU table
# ---------------------------------------------
# Expect a TSV file with ASVs as rows and sample IDs as columns
# Example: asv_table.tsv (tab-separated)

ASV_TABLE = 'asv_table.tsv'  # change if needed

otu = pd.read_csv(ASV_TABLE, sep='	', index_col=0)
print('Loaded table with shape:', otu.shape)
display(otu.head())


Loaded table with shape: (9000, 12)


,soil A,soil B,soil C,soil D,viable cells A,viable cells B,viable cells C,viable cells D,Live-FISH A,Live-FISH B,Live-FISH C,Live-FISH D
ASV1,66,70,35,52,28,32,15,19,31,24,38,32
ASV2,4045,3301,3437,3453,9025,8260,10216,9806,373,425,318,546
ASV3,112,110,84,121,1059,1163,1060,798,9029,10576,11768,4041
ASV4,2352,2990,2767,2647,912,924,455,2095,6043,5646,3172,9207
ASV5,35,27,48,66,2816,2275,2190,192,6188,5451,6171,346


In [3]:
# ---------------------------------------------
# Alpha diversity metrics
# ---------------------------------------------
def richness(counts: np.ndarray) -> int:
    """Observed features = number of non-zero ASVs.
    counts: 1D array of counts for a sample
    """
    return int(np.sum(counts > 0))

def shannon(counts: np.ndarray) -> float:
    """Shannon index (natural log). Zero counts are ignored.
    """
    total = counts.sum()
    if total <= 0:
        return 0.0
    p = counts[counts > 0] / total
    return float(-np.sum(p * np.log(p)))

observed_features = {}
shannon_index = {}

for sample in otu.columns:
    vals = otu[sample].to_numpy()
    observed_features[sample] = richness(vals)
    shannon_index[sample] = shannon(vals)
    pielou_index[sample] = pielou(vals)

observed_features = pd.Series(observed_features)
shannon_index = pd.Series(shannon_index)

print('Observed features (first 10):')
display(observed_features.head(10))


Observed features (first 10):


soil A            1993
soil B            2183
soil C            1784
soil D            1794
viable cells A    1597
viable cells B    1736
viable cells C    1613
viable cells D    1924
Live-FISH A       1569
Live-FISH B       1620
dtype: int64

Shannon (first 10):


soil A            6.023421
soil B            6.038031
soil C            5.947271
soil D            6.090065
viable cells A    5.650723
viable cells B    5.673919
viable cells C    5.608601
viable cells D    5.837191
Live-FISH A       5.458049
Live-FISH B       5.452189
dtype: float64

Pielou (first 10):


soil A            0.792827
soil B            0.785337
soil C            0.794387
soil D            0.812854
viable cells A    0.766108
viable cells B    0.760646
viable cells C    0.759371
viable cells D    0.771895
Live-FISH A       0.741765
Live-FISH B       0.737761
dtype: float64

In [5]:
# ---------------------------------------------
# Plot: Observed Richness (bar)
# ---------------------------------------------
# Assumes 3 groups of 4 samples; adjust group sizes/labels to your design.
values = observed_features.values
n = len(values)
if n < 12:
    print('Warning: fewer than 12 samples; grouping code may need adjustment.')

group1 = values[0:4]
group2 = values[4:8]
group3 = values[8:12]
data = np.concatenate([group1, group2, group3])
colors = ['navy']*4 + ['green']*4 + ['orange']*4

plt.figure(figsize=(10,5))
plt.bar(range(len(data)), data, color=colors)
plt.xticks([1.5, 5.5, 9.5], ['soil', 'viable cells', 'viable cells after Live-FISH'])
plt.title('Observed Richness')
plt.ylabel('Number of ASVs')
plt.xlabel('Treatment')
plt.tight_layout()
plt.savefig('observed_richness.png', dpi=300)
plt.close()
print('Saved: observed_richness.png')


Saved: observed_richness.png


In [6]:
# ---------------------------------------------
# Plot: Shannon Diversity (boxplot)
# ---------------------------------------------
values = shannon_index.values
group1 = values[0:4]
group2 = values[4:8]
group3 = values[8:12]
data = [group1, group2, group3]
colors = ['navy', 'green', 'orange']

plt.figure(figsize=(8,5))
bp = plt.boxplot(data, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
plt.xticks([1,2,3], ['soil', 'viable cells', 'viable cells after Live-FISH'])
plt.title('Shannon Diversity Boxplot')
plt.ylabel("Shannon (H')")
plt.tight_layout()
plt.savefig('shannon_diversity_boxplot.png', dpi=300)
plt.close()
print('Saved: shannon_diversity_boxplot.png')


Saved: shannon_diversity_boxplot.png


In [7]:
# ---------------------------------------------
# Beta diversity: Bray–Curtis + PCoA
# ---------------------------------------------
# Samples as rows
X = otu.T.to_numpy()  # shape = (n_samples, n_features)
sample_ids = otu.columns.to_list()

def bray_curtis_numpy(X: np.ndarray) -> np.ndarray:
    """Compute pairwise Bray–Curtis dissimilarity using NumPy.
    Returns an (n_samples x n_samples) matrix.
    """
    eps = 1e-12  # avoid division by zero when rows are all zeros
    n = X.shape[0]
    D = np.zeros((n, n), dtype=float)
    for i in range(n):
        Xi = X[i]
        for j in range(i+1, n):
            Xj = X[j]
            num = np.sum(np.abs(Xi - Xj))
            den = np.sum(Xi + Xj) + eps
            d = num / den
            D[i, j] = D[j, i] = d
    return D

if HAVE_SKLEARN:
    D = pairwise_distances(X, metric='braycurtis')
else:
    D = bray_curtis_numpy(X)

# Classical MDS / PCoA via eigen-decomposition to obtain variance explained
def pcoa_from_distance(D: np.ndarray, k: int = 2):
    """Perform PCoA using classical scaling.
    Returns coordinates (n x k) and proportion explained per axis.
    """
    n = D.shape[0]
    J = np.eye(n) - np.ones((n, n)) / n
    D2 = D ** 2
    B = -0.5 * J @ D2 @ J
    w, V = np.linalg.eigh(B)  # ascending order
    pos = w > 1e-12
    w_pos = w[pos]
    V_pos = V[:, pos]
    idx = np.argsort(w_pos)[::-1]
    w_sorted = w_pos[idx]
    V_sorted = V_pos[:, idx]
    L = np.diag(np.sqrt(w_sorted[:k]))
    coords = V_sorted[:, :k] @ L
    prop = w_sorted / np.sum(w_sorted) if np.sum(w_sorted) > 0 else np.zeros_like(w_sorted)
    return coords, prop

coords, prop = pcoa_from_distance(D, k=2)
pc = pd.DataFrame(coords, index=sample_ids, columns=['PC1', 'PC2'])
pc1_var = float(prop[0] * 100) if len(prop) > 0 else float('nan')
pc2_var = float(prop[1] * 100) if len(prop) > 1 else float('nan')

print('PCoA head:')
display(pc.head())
print('Explained variance (%): PC1=', round(pc1_var, 2), ' PC2=', round(pc2_var, 2))


PCoA head:


,PC1,PC2
soil A,-0.350819,0.187218
soil B,-0.327545,0.220704
soil C,-0.352767,0.195391
soil D,-0.339991,0.199778
viable cells A,-0.061379,-0.343413


Explained variance (%): PC1= 57.81  PC2= 31.28


In [8]:
# ---------------------------------------------
# Plot: PCoA scatter (saved only)
# ---------------------------------------------
def color_for(sample_name: str) -> str:
    s = str(sample_name).lower()
    if s.startswith('soil'):
        return 'navy'
    if s.startswith('viable cells'):
        return 'green'
    if s.startswith('live-fish') or s.startswith('live_fish') or s.startswith('live fish'):
        return 'orange'
    return 'gray'

colors = [color_for(s) for s in pc.index]
label_order = [('soil', 'navy'), ('viable cells', 'green'), ('Live-FISH', 'orange')]
present = [(lab, col) for lab, col in label_order if col in colors]

plt.figure(figsize=(6,5))
plt.scatter(pc['PC1'], pc['PC2'], s=60, edgecolor='k', c=colors)
for sample in pc.index:
    plt.text(pc.loc[sample, 'PC1'], pc.loc[sample, 'PC2'], sample, fontsize=8, va='bottom')
for lab, col in present:
    plt.scatter([], [], c=col, edgecolor='k', s=60, label=lab)
plt.legend(frameon=True, title='Group', loc='best')
plt.xlabel(f"PC1 ({pc1_var:.1f}%)")
plt.ylabel(f"PC2 ({pc2_var:.1f}%)")
plt.title('PCoA (Bray–Curtis)')
plt.tight_layout()
plt.savefig('pcoa_braycurtis.png', dpi=300)
plt.close()
print('Saved: pcoa_braycurtis.png')


Saved: pcoa_braycurtis.png
